In [1]:
from pathlib import Path
import hashlib
import json
import shutil

import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

log_path = RAW_DIR / "user_logs.csv"
output_dir = PROCESSED_DIR / "listening_windows_30d"
checkpoint_path = output_dir / "_SUCCESS.json"

temp_dir = PROJECT_ROOT / "data" / "duckdb_tmp"
temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(config={
    "threads": 2,
    "memory_limit": "2GB",
    "temp_directory": str(temp_dir),
    "max_temp_directory_size": "2GB",
})

assert log_path.exists(), "Raw listening logs not found."

In [2]:
training_users = con.read_parquet(
    str(PROCESSED_DIR / "historical_labels_dec2016.parquet")
).project("msno").df()

evaluation_users = con.read_parquet(
    str(
        PROCESSED_DIR
        / "candidate_transaction_features_v2_cutoff_20170228.parquet"
    )
).project("msno").df()

required_windows = pd.concat([
    training_users.assign(
        cohort="historical_train",
        window_start=pd.Timestamp("2016-11-01"),
        window_end=pd.Timestamp("2016-11-30"),
    ),
    evaluation_users.assign(
        cohort="later_evaluation",
        window_start=pd.Timestamp("2017-01-30"),
        window_end=pd.Timestamp("2017-02-28"),
    ),
], ignore_index=True)

assert not required_windows.duplicated(["cohort", "msno"]).any()

con.register("required_windows", required_windows)

required_windows.groupby("cohort").agg(
    users=("msno", "size"),
    window_start=("window_start", "min"),
    window_end=("window_end", "max"),
)

,users,window_start,window_end
cohort,,,
historical_train,882127,2016-11-01,2016-11-30
later_evaluation,970960,2017-01-30,2017-02-28


In [3]:
count_columns = [
    "num_25",
    "num_50",
    "num_75",
    "num_985",
    "num_100",
    "num_unq",
]

count_casts = ",\n".join(
    f"TRY_CAST({column} AS BIGINT) AS {column}"
    for column in count_columns
)

count_selections = ", ".join(
    f"p.{column}"
    for column in count_columns
)

extraction_sql = f"""
WITH parsed AS (
    SELECT
        msno,
        TRY_STRPTIME("date", '%Y%m%d')::DATE AS log_day,
        {count_casts},
        TRY_CAST(total_secs AS DOUBLE) AS total_secs_raw
    FROM read_csv(?, all_varchar = true)
)
SELECT
    w.cohort,
    p.msno,
    p.log_day,
    {count_selections},
    p.total_secs_raw,
    CASE
        WHEN ISFINITE(p.total_secs_raw)
         AND p.total_secs_raw BETWEEN 0 AND 86400
        THEN p.total_secs_raw
        ELSE NULL
    END AS total_secs_clean,
    CASE
        WHEN ISFINITE(p.total_secs_raw)
         AND p.total_secs_raw BETWEEN 0 AND 86400
        THEN 0
        ELSE 1
    END AS duration_anomaly
FROM parsed AS p
INNER JOIN required_windows AS w
    ON p.msno = w.msno
   AND p.log_day BETWEEN w.window_start AND w.window_end
"""

In [4]:
source_stat = log_path.stat()

extraction_config = {
    "sql_sha256": hashlib.sha256(
        extraction_sql.encode("utf-8")
    ).hexdigest(),
    "source_size_bytes": source_stat.st_size,
    "source_mtime_ns": source_stat.st_mtime_ns,
    "historical_users": len(training_users),
    "evaluation_users": len(evaluation_users),
    "training_window": ["2016-11-01", "2016-11-30"],
    "evaluation_window": ["2017-01-30", "2017-02-28"],
}

if checkpoint_path.exists():
    saved_config = json.loads(
        checkpoint_path.read_text(encoding="utf-8")
    )
    assert saved_config == extraction_config, (
        "Existing extraction configuration differs from this run."
    )
    print("Reusing the completed extraction.")
else:
    assert not output_dir.exists(), (
        "An output directory exists without a completion checkpoint. "
        "Inspect it before rerunning extraction."
    )

    assert shutil.disk_usage(PROCESSED_DIR).free > 4 * 1024**3, (
        "Less than 4 GiB of working disk space is available."
    )

    print("Extracting listening windows to Parquet...", flush=True)

    con.sql(
        extraction_sql,
        params=[str(log_path)],
    ).write_parquet(
        str(output_dir),
        compression="zstd",
        partition_by=["cohort"],
        row_group_size=100000,
    )

    checkpoint_path.write_text(
        json.dumps(extraction_config, indent=2) + "\n",
        encoding="utf-8",
    )

    print("Extraction complete.")

Extracting listening windows to Parquet...
Extraction complete.


In [5]:
parquet_pattern = str(output_dir / "**" / "*.parquet")

window_summary = con.execute(
    """
    SELECT
        cohort,
        COUNT(*) AS rows,
        COUNT(DISTINCT msno) AS users_with_logs,
        MIN(log_day) AS first_log_day,
        MAX(log_day) AS last_log_day,
        SUM(duration_anomaly) AS flagged_duration_rows
    FROM read_parquet(?, hive_partitioning = true)
    GROUP BY cohort
    ORDER BY cohort
    """,
    [parquet_pattern],
).df()

parquet_bytes = sum(
    path.stat().st_size
    for path in output_dir.rglob("*.parquet")
)

print(f"Parquet size: {parquet_bytes / 1024**3:.2f} GiB")
print(
    f"Remaining disk space: "
    f"{shutil.disk_usage(PROCESSED_DIR).free / 1024**3:.2f} GiB"
)

window_summary

Parquet size: 1.04 GiB
Remaining disk space: 10.24 GiB


,cohort,rows,users_with_logs,first_log_day,last_log_day,flagged_duration_rows
0,historical_train,11326275,667477,2016-11-01,2016-11-30,2384.0
1,later_evaluation,12682786,758941,2017-01-30,2017-02-28,3052.0


## Check Duplicate User-Day Records

Check each cohort separately before constructing daily features
or model sequences. Multiple records on the same day are not
automatically summed or removed.

In [6]:
duplicate_audit_sql = """
WITH duplicate_days AS (
    SELECT
        msno,
        log_day,
        COUNT(*) AS records_per_day
    FROM read_parquet(?, hive_partitioning = true)
    WHERE cohort = ?
    GROUP BY msno, log_day
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS duplicate_user_days,
    COALESCE(
        SUM(records_per_day), 0
    ) AS rows_in_duplicate_days,
    COALESCE(
        SUM(records_per_day - 1), 0
    ) AS excess_rows,
    COALESCE(
        MAX(records_per_day), 0
    ) AS max_records_per_duplicate_day
FROM duplicate_days
"""

duplicate_results = []

for cohort in ["historical_train", "later_evaluation"]:
    print(f"Checking {cohort}...", flush=True)

    result = con.execute(
        duplicate_audit_sql,
        [parquet_pattern, cohort],
    ).df()

    result.insert(0, "cohort", cohort)
    duplicate_results.append(result)

daily_duplicate_audit = pd.concat(
    duplicate_results,
    ignore_index=True,
)

daily_duplicate_audit

Checking historical_train...
Checking later_evaluation...


,cohort,duplicate_user_days,rows_in_duplicate_days,excess_rows,max_records_per_duplicate_day
0,historical_train,0,0.0,0.0,0
1,later_evaluation,0,0.0,0.0,0


In [7]:
report_dir = PROJECT_ROOT / "reports"
sql_dir = PROJECT_ROOT / "sql"

report_dir.mkdir(parents=True, exist_ok=True)
sql_dir.mkdir(parents=True, exist_ok=True)

assert daily_duplicate_audit["duplicate_user_days"].eq(0).all()

expected_windows = {
    "historical_train": (
        pd.Timestamp("2016-11-01"),
        pd.Timestamp("2016-11-30"),
    ),
    "later_evaluation": (
        pd.Timestamp("2017-01-30"),
        pd.Timestamp("2017-02-28"),
    ),
}

assert set(window_summary["cohort"]) == set(expected_windows)

for row in window_summary.itertuples(index=False):
    start, end = expected_windows[row.cohort]

    assert pd.Timestamp(row.first_log_day) >= start
    assert pd.Timestamp(row.last_log_day) <= end

window_summary.to_csv(
    report_dir / "listening_window_summary.csv",
    index=False,
)

daily_duplicate_audit.to_csv(
    report_dir / "listening_user_day_duplicates.csv",
    index=False,
)

(sql_dir / "extract_listening_windows.sql").write_text(
    extraction_sql.strip() + "\n",
    encoding="utf-8",
)

(sql_dir / "listening_user_day_duplicates.sql").write_text(
    duplicate_audit_sql.strip() + "\n",
    encoding="utf-8",
)

daily_data_config = {
    **extraction_config,
    "output_directory": "data/processed/listening_windows_30d",
    "partition_column": "cohort",
    "compression": "zstd",
    "window_days": 30,
    "user_day_duplicates_checked": True,
    "duplicate_user_days": 0,
    "duration_upper_bound": 86400,
    "raw_duration_retained": True,
    "clean_duration_column": "total_secs_clean",
    "duration_anomaly_column": "duration_anomaly",
    "threshold_sensitivity_checked": False,
    "missing_user_days_materialized": False,
}

(report_dir / "daily_listening_config.json").write_text(
    json.dumps(daily_data_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Daily listening checks, SQL, and configuration saved.")

Daily listening checks, SQL, and configuration saved.


## Daily Listening Data Preparation

Extracted two 30-day listening windows for the modeling cohorts and saved them as Zstandard-compressed Parquet partitions.

| Cohort | Window | Records | Users with logs | Users without window logs |
|---|---|---:|---:|---:|
| Historical training | Nov 1–30, 2016 | 11,326,275 | 667,477 | 214,650 |
| Later evaluation | Jan 30–Feb 28, 2017 | 12,682,786 | 758,941 | 212,019 |

No duplicate user-day records were found in either extracted window. No deduplication or within-day aggregation was required.

The extracted data occupies approximately 1.04 GiB, compared with the 28.42 GiB raw CSV. User-level data remains local and is excluded from Git.

Song counts, raw listening duration, cleaned duration, and a duration anomaly indicator are retained. The provisional duration threshold still requires sensitivity checks.

Users and days without logs are not yet materialized. Coverage indicators and missing-day handling will be added during feature and sequence construction.

See [the preparation notebook](notebooks/10_daily_listening_data.ipynb), [window summary](reports/listening_window_summary.csv), [duplicate checks](reports/listening_user_day_duplicates.csv), and [configuration](reports/daily_listening_config.json).